## **Analysing Feature Importance for answering the questions why customers churn and which are the most important features to answer this**

In [2]:
import pandas as pd
from pathlib import Path
import re
import json
from dotenv import load_dotenv
import os
import  google.generativeai as genai

In [3]:
script = Path.cwd()
root = script.parent
feature_importance = pd.read_csv(root/"data"/"cleaned_data"/'feature_importance.csv')
feature_importance.to_string
data = pd.read_csv(root/"data"/"cleaned_data"/'clean_clustered.csv')

In [4]:
# Removing extra stuffs from the feature names 
def map_to_original(feat_name, original_columns):
    clean = re.sub(r'^(num__|cat__)', '', feat_name)
    for col in original_columns:
        if clean.startswith(col):
            return col
    return clean

In [5]:
original_columns = ['avg_time_spent','days_since_last_login','avg_transaction_value',
                    'preferred_offer_types','used_special_discount','internet_option',
                    'last_visit_time','joining_date','avg_frequency_login_days','region_category']


In [6]:
feature_importance['feature'] = feature_importance['Feature'].apply(
    lambda x: map_to_original(x, original_columns)
)

In [7]:
feature_importance.head()

,Feature,Importance,feature
0,num__age,0.369645,age
1,cat__joined_through_referral,0.287404,joined_through_referral
2,cat__feedback,0.053020,feedback
3,num__points_in_wallet,0.043710,points_in_wallet
4,cat__offer_application_preference,0.025449,offer_application_preference


In [9]:
importance = feature_importance.groupby('feature')['Importance'].sum().sort_values(ascending=False).reset_index()
importance.to_csv(root/"data"/"cleaned_data"/"feature_importance.csv")
print(f"Successfully Saved feature_importance to {root/"data"/"cleaned_data"/"feature_importance.csv"} ✅")

Successfully Saved feature_importance to d:\Customer Segmntation and retention analysis\data\cleaned_data\feature_importance.csv ✅


In [8]:
top_n = 3
top_features = importance.head(top_n).index.tolist()
print("Top churn-driving features:", top_features)

Top churn-driving features: ['age', 'joined_through_referral', 'feedback']


In [9]:

for feature in top_features:
    print(f"\n{'='*60}")
    print(f"CHURN RATE BY: {feature}")
    print(f"{'='*60}")

    churn_by_group = data.groupby(feature)['churn_risk_score'].agg(
        total_customers='count',
        churned_customers='sum'
    )
    churn_by_group['churn_rate_%'] = (churn_by_group['churned_customers'] / churn_by_group['total_customers'] * 100).round(2)
    churn_by_group = churn_by_group.sort_values(by='churn_rate_%', ascending=False)
    churn_by_group.to_csv(root/"data"/"feature_importance"/f"{feature}_importance.csv")

    
    print(churn_by_group.to_string())


CHURN RATE BY: age
      total_customers  churned_customers  churn_rate_%
age                                                   
37.0             2334             2067.0         88.56
38.0              674              390.0         57.86
28.0              657              376.0         57.23
62.0              653              372.0         56.97
56.0              655              369.0         56.34
35.0              631              355.0         56.26
29.0              646              361.0         55.88
59.0              666              370.0         55.56
52.0              639              354.0         55.40
31.0              605              335.0         55.37
54.0              586              324.0         55.29
11.0              626              346.0         55.27
49.0              630              348.0         55.24
25.0              602              332.0         55.15
33.0              687              376.0         54.73
50.0              625              342.0     

# Key Takeaway : 
#### Age Group from 37-40 are the most churners
#### Persons who joined through referral are most churners
#### Feedback of people 'Poor Product Quality' have mostly churned!

In [4]:
churn_findings = {
    "overall_churn_rate_prediction_by_model" : "56.30%",
    "model_accuracy": "93.33%",
    "top_churn_drivers": [
        {"feature": "age", "importance": "36.9%", 
        "insight": "Customers aged 37-40 churn the most"},
        {"feature": "joined_through_referral", "importance": "28.7%", 
        "insight": "Customers who joined via referral churn more than direct signups"},
        {"feature": "feedback", "importance": "5.3%", 
        "insight": "'Poor Product Quality' feedback correlates with 64.81% churn rate"},
        {"feature" : "feedback","label" : "No data available" ,"churn" : "100%",
        "insight" : "No data available for customers who have not given feedback churned highest!"}
        ,{"feature" : "joined_through_referral","label" : "No data available","churn" : "100%",
        "insight" : "No data available for joined_through_referral customers churned highest!"}
    ],
    "churn_by_feedback_category": {
        "Poor Product Quality": "64.81%",
        "Poor Customer Service": "63.80%",
        "No reason specified": "63.43%",
        "Poor Website": "62.97%",
        "Too many ads": "62.70%",
        "Products always in Stock": "0.00%",
        "Quality Customer Care": "0.00%",
        "Reasonable Price": "0.00%",
        "User Friendly Website": "0.00%"
    }

}

In [5]:
prompt = f"""
You are a senior customer retention analyst. Based STRICTLY on the data below 
(do not invent numbers not present here), write a structured churn analysis report.

DATA:
{json.dumps(churn_findings, indent=2)}

Structure your report EXACTLY as follows, using markdown headers:

## Executive Summary
(2-3 sentences: overall churn scale and biggest driver)

## Top Churn Risk Factors
(Ranked list, explain WHY each factor likely drives churn in this business context)

## Customer Segments at Highest Risk
(Based on the data, define 2-3 concrete customer segments to prioritize)

## Retention Recommendations
(For EACH segment above, give 2-3 specific, actionable retention tactics — 
not generic advice. Tie each tactic directly back to the data point that justifies it)

## Suggested Next Steps for the Data Team
(What additional data or analysis would sharpen this further)

Keep the tone professional, concise, and business-ready. No fluff.
"""

In [6]:
from pathlib import Path
from dotenv import load_dotenv
import os

# Explicitly load BOTH .env files by their real, fixed locations
root_env = Path.cwd().resolve().parent/ ".env"      # adjust .parent count to match your folder depth
load_dotenv(dotenv_path=root_env)
api_key = os.getenv("GEMINI_API_KEY")
MODEL_NAME="gemini-3.5-flash-lite"
model = genai.GenerativeModel(MODEL_NAME)

In [7]:
response = model.generate_content(prompt)
with open(root/"models"/"reports"/"response.txt", "w", encoding="utf-8") as f:
    f.write(response.text)

print("Saved to response.txt")

Saved to response.txt


## A brief intro for the important feature analysis

In [20]:
print(response.text)

## Executive Summary
The predictive model indicates a critical overall churn rate of 56.30%, with a high model accuracy of 93.33%. Customer age emerges as the single most influential churn driver, accounting for 36.9% of the model's importance. 

## Top Churn Risk Factors
1. **Age (36.9% importance):** Customers aged 37-40 churn the most, indicating that this specific life-stage or demographic cohort faces unmet needs or misaligned product-market fit.
2. **Joined Through Referral (28.7% importance):** Customers who joined via referral churn more than direct signups, suggesting that referral acquisition channels may be attracting misaligned incentives or mismatched expectations compared to direct acquisition.
3. **Feedback / Product Quality (5.3% importance):** "Poor Product Quality" feedback strongly correlates with a 64.81% churn rate, pointing to core product functionality or reliability issues driving users away. Furthermore, missing data flags for feedback and referral status exhib

In [9]:
from fpdf import FPDF
import re

def txt_to_pdf(input_path, output_path, title="Document"):
    with open(input_path, "r", encoding="utf-8") as f:
        text = f.read()

    pdf = FPDF(format="A4")
    pdf.set_auto_page_break(auto=True, margin=20)
    pdf.add_page()

    # Title
    pdf.set_font("Helvetica", "B", 16)
    pdf.multi_cell(0, 10, title)
    pdf.ln(4)

    # Split into paragraphs on blank lines
    paragraphs = re.split(r"\n\s*\n", text.strip())

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue

        # Simple heuristic: short line, no ending punctuation -> treat as a heading
        is_heading = len(para) < 80 and not para.endswith((".", ",", ":", ";")) and "\n" not in para

        if is_heading:
            pdf.set_font("Helvetica", "B", 13)
            pdf.multi_cell(0, 8, para)
            pdf.ln(2)
        else:
            pdf.set_font("Helvetica", "", 11)
            # Preserve line breaks within a paragraph (bullet-like lines)
            for line in para.split("\n"):
                pdf.multi_cell(0, 6, line)
            pdf.ln(4)

    pdf.output(output_path)
    print(f"Saved: {output_path}")

# Usage
txt_to_pdf(root/"models"/"reports"/"response.txt",root/"models"/"reports"/"response.pdf", title="Gemini Response")

Saved: d:\Customer Segmntation and retention analysis\models\reports\response.pdf
